# 解析 response_body 平铺展示

本 Notebook 用于解析 `cdc_pickle_pass_fpd7.pkl` 文件，将 `response_body` 中的 JSON 数据平铺成表格形式展示。

## 功能说明

1. 读取 pickle 文件
2. 解析 response_body 的 JSON 结构
3. 将四个子板块（consultas、creditos、empleos、domicilios）分别平铺成表格
4. 可选：输出到 Excel 文件

## 输出内容

- **consultas_<timestamp>.xlsx**：查询记录明细表
- **creditos_<timestamp>.xlsx**：信贷账户明细表
- **empleos_<timestamp>.xlsx**：工作记录明细表
- **domicilios_<timestamp>.xlsx**：住址记录明细表
- **summary_<timestamp>.xlsx**：汇总统计表

## 1. 导入库

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

print("[INFO] 库导入成功")

## 2. 配置参数

In [ ]:
# 配置参数
PICKLE_PATH = Path("cdc_pickle_pass_fpd7.pkl")
OUTPUT_DIR = Path("outputs/response_body_flat")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 是否输出到 Excel（True=输出；False=只在内存中查看）
WRITE_EXCEL = False

# 是否输出到 CSV（True=输出；False=只在内存中查看）
WRITE_CSV = True

# 最大处理样本数（None=全部处理，数字=只处理前N个）
MAX_SAMPLES = None  # 改为 10 可以快速测试

print(f"[INFO] 配置完成")
print(f"  - 输入文件: {PICKLE_PATH}")
print(f"  - 输出目录: {OUTPUT_DIR}")
print(f"  - 输出 Excel: {WRITE_EXCEL}")
print(f"  - 输出 CSV: {WRITE_CSV}")
print(f"  - 最大样本数: {MAX_SAMPLES if MAX_SAMPLES else '全部'}")

## 3. 读取数据

In [ ]:
# 读取 pickle 文件
print("[INFO] 正在读取 pickle 文件...")
df_raw = pd.read_pickle(PICKLE_PATH)

# 统一字段名：apply_time -> request_time
if "request_time" not in df_raw.columns and "apply_time" in df_raw.columns:
    df_raw["request_time"] = df_raw["apply_time"]
    print("[INFO] 已映射 apply_time -> request_time")

print(f"[INFO] 读取完成，共 {len(df_raw):,} 条记录")
print(f"[INFO] 字段列表: {list(df_raw.columns)}")

# 显示前几行
df_raw.head(3)
df_raw

## 4. 定义解析函数

In [ ]:
def parse_response_body(df: pd.DataFrame, max_samples: int = None):
    """
    解析 response_body，提取四个子板块的数据
    
    参数：
        df: 原始数据 DataFrame
        max_samples: 最大处理样本数（None=全部处理）
    
    返回：
        dict: {
            'consultas': DataFrame,
            'creditos': DataFrame,
            'empleos': DataFrame,
            'domicilios': DataFrame,
            'summary': DataFrame
        }
    """
    
    consultas_rows = []
    creditos_rows = []
    empleos_rows = []
    domicilios_rows = []
    summary_rows = []
    
    print('[INFO] 开始解析 response_body...')
    
    total = len(df) if max_samples is None else min(len(df), max_samples)
    
    for idx, row in df.iterrows():
        if max_samples and idx >= max_samples:
            break
            
        if idx % 1000 == 0:
            print(f'  处理进度: {idx}/{total}')
        
        apply_id = row.get('apply_id')
        request_time = row.get('request_time')
        response_body = row.get('response_body')
        
        summary = {
            'apply_id': apply_id,
            'request_time': request_time,
            'consultas_cnt': 0,
            'creditos_cnt': 0,
            'empleos_cnt': 0,
            'domicilios_cnt': 0,
            'parse_status': 'ok'
        }
        
        # 检查 response_body 是否为空
        if pd.isna(response_body) or response_body is None or str(response_body).strip() == '':
            summary['parse_status'] = 'empty_response_body'
            summary_rows.append(summary)
            continue
        
        # 解析 JSON
        try:
            if isinstance(response_body, str):
                obj = json.loads(response_body)
            else:
                obj = response_body
        except Exception as e:
            summary['parse_status'] = f'json_error: {str(e)[:50]}'
            summary_rows.append(summary)
            continue
        
        if not isinstance(obj, dict):
            summary['parse_status'] = 'not_dict'
            summary_rows.append(summary)
            continue
        
        # 解析 consultas
        consultas = obj.get('consultas', [])
        if isinstance(consultas, list):
            summary['consultas_cnt'] = len(consultas)
            for item in consultas:
                if isinstance(item, dict):
                    consultas_rows.append({
                        'apply_id': apply_id,
                        'request_time': request_time,
                        **item
                    })
        
        # 解析 creditos
        creditos = obj.get('creditos', [])
        if isinstance(creditos, list):
            summary['creditos_cnt'] = len(creditos)
            for item in creditos:
                if isinstance(item, dict):
                    creditos_rows.append({
                        'apply_id': apply_id,
                        'request_time': request_time,
                        **item
                    })
        
        # 解析 empleos
        empleos = obj.get('empleos', [])
        if isinstance(empleos, list):
            summary['empleos_cnt'] = len(empleos)
            for item in empleos:
                if isinstance(item, dict):
                    empleos_rows.append({
                        'apply_id': apply_id,
                        'request_time': request_time,
                        **item
                    })
        
        # 解析 domicilios
        domicilios = obj.get('domicilios', [])
        if isinstance(domicilios, list):
            summary['domicilios_cnt'] = len(domicilios)
            for item in domicilios:
                if isinstance(item, dict):
                    domicilios_rows.append({
                        'apply_id': apply_id,
                        'request_time': request_time,
                        **item
                    })
        
        summary_rows.append(summary)
    
    print('[INFO] 解析完成')
    
    # 转换为 DataFrame
    result = {
        'consultas': pd.DataFrame(consultas_rows),
        'creditos': pd.DataFrame(creditos_rows),
        'empleos': pd.DataFrame(empleos_rows),
        'domicilios': pd.DataFrame(domicilios_rows),
        'summary': pd.DataFrame(summary_rows)
    }
    
    return result

print("[INFO] 解析函数定义完成")

## 5. 执行解析

In [ ]:
# 执行解析
parsed_data = parse_response_body(df_raw, max_samples=MAX_SAMPLES)

# 显示统计信息
print("\n" + "="*60)
print("解析结果统计")
print("="*60)
print(f"consultas（查询记录）: {len(parsed_data['consultas']):,} 条")
print(f"creditos（信贷账户）: {len(parsed_data['creditos']):,} 条")
print(f"empleos（工作记录）: {len(parsed_data['empleos']):,} 条")
print(f"domicilios（住址记录）: {len(parsed_data['domicilios']):,} 条")
print("="*60)

## 6. 查看汇总表

In [ ]:
# 显示汇总表
print("汇总表（前 10 条）：")
parsed_data["summary"].head(10)

## 7. 查看各板块数据

### 7.1 consultas（查询记录）

In [ ]:
print("\n" + "="*60)
print("consultas（查询记录）")
print("="*60)
print(f"记录数: {len(parsed_data['consultas']):,}")
print(f"字段数: {len(parsed_data['consultas'].columns)}")
print(f"\n字段列表:")
for i, col in enumerate(parsed_data['consultas'].columns, 1):
    print(f"  {i:2d}. {col}")

print("\n前 5 条记录:")
parsed_data["consultas"].head(5)

#### 查询某个 apply_id 的 consultas 明细

In [ ]:
# 查询某个 apply_id 的 consultas 明细
# 修改下面的 apply_id 来查询不同的申请
query_apply_id = '1065991091661283329'  # 修改这里

consultas_detail = parsed_data['consultas'][parsed_data['consultas']['apply_id'] == query_apply_id]

print(f"apply_id = {query_apply_id} 的 consultas 明细")
print("="*60)
print(f"共 {len(consultas_detail)} 条查询记录")
print()

if len(consultas_detail) > 0:
    # 显示所有记录
    consultas_detail
else:
    print("未找到该 apply_id 的查询记录")

### 7.2 creditos（信贷账户）

In [ ]:
print("\n" + "="*60)
print("creditos（信贷账户）")
print("="*60)
print(f"记录数: {len(parsed_data['creditos']):,}")
print(f"字段数: {len(parsed_data['creditos'].columns)}")
print(f"\n字段列表:")
for i, col in enumerate(parsed_data['creditos'].columns, 1):
    print(f"  {i:2d}. {col}")

print("\n前 5 条记录:")
parsed_data["creditos"].head(5)

#### 查询某个 apply_id 的 creditos 明细

In [ ]:
# 查询某个 apply_id 的 creditos 明细
# 修改下面的 apply_id 来查询不同的申请
query_apply_id = '1065991091661283329'  # 修改这里

creditos_detail = parsed_data['creditos'][parsed_data['creditos']['apply_id'] == query_apply_id]

print(f"apply_id = {query_apply_id} 的 creditos 明细")
print("="*60)
print(f"共 {len(creditos_detail)} 个信贷账户")
print()

if len(creditos_detail) > 0:
    # 显示所有记录
    creditos_detail
else:
    print("未找到该 apply_id 的信贷账户")

### 7.3 empleos（工作记录）

In [ ]:
print("\n" + "="*60)
print("empleos（工作记录）")
print("="*60)
print(f"记录数: {len(parsed_data['empleos']):,}")
print(f"字段数: {len(parsed_data['empleos'].columns)}")
print(f"\n字段列表:")
for i, col in enumerate(parsed_data['empleos'].columns, 1):
    print(f"  {i:2d}. {col}")

print("\n前 5 条记录:")
parsed_data["empleos"].head(5)

#### 查询某个 apply_id 的 empleos 明细

In [ ]:
# 查询某个 apply_id 的 empleos 明细
# 修改下面的 apply_id 来查询不同的申请
query_apply_id = '1065991091661283329'  # 修改这里

empleos_detail = parsed_data['empleos'][parsed_data['empleos']['apply_id'] == query_apply_id]

print(f"apply_id = {query_apply_id} 的 empleos 明细")
print("="*60)
print(f"共 {len(empleos_detail)} 条工作记录")
print()

if len(empleos_detail) > 0:
    # 显示所有记录
    empleos_detail
else:
    print("未找到该 apply_id 的工作记录")

### 7.4 domicilios（住址记录）

In [ ]:
print("\n" + "="*60)
print("domicilios（住址记录）")
print("="*60)
print(f"记录数: {len(parsed_data['domicilios']):,}")
print(f"字段数: {len(parsed_data['domicilios'].columns)}")
print(f"\n字段列表:")
for i, col in enumerate(parsed_data['domicilios'].columns, 1):
    print(f"  {i:2d}. {col}")

print("\n前 5 条记录:")
parsed_data["domicilios"].head(5)

#### 查询某个 apply_id 的 domicilios 明细

In [ ]:
# 查询某个 apply_id 的 domicilios 明细
# 修改下面的 apply_id 来查询不同的申请
query_apply_id = '1065991091661283329'  # 修改这里

domicilios_detail = parsed_data['domicilios'][parsed_data['domicilios']['apply_id'] == query_apply_id]

print(f"apply_id = {query_apply_id} 的 domicilios 明细")
print("="*60)
print(f"共 {len(domicilios_detail)} 条住址记录")
print()

if len(domicilios_detail) > 0:
    # 显示所有记录
    domicilios_detail
else:
    print("未找到该 apply_id 的住址记录")

## 8. 快速查询示例

### 8.1 查看某个 apply_id 的所有查询记录

In [ ]:
# 示例1：查看某个 apply_id 的所有查询记录
sample_apply_id = parsed_data["consultas"]["apply_id"].iloc[0] if len(parsed_data["consultas"]) > 0 else None

if sample_apply_id:
    print(f"示例1：查看 apply_id={sample_apply_id} 的所有查询记录")
    print("="*60)
    sample_consultas = parsed_data["consultas"][parsed_data["consultas"]["apply_id"] == sample_apply_id]
    print(f"共 {len(sample_consultas)} 条查询记录")
    sample_consultas

### 8.2 查看某个 apply_id 的所有信贷账户

In [ ]:
# 示例2：查看某个 apply_id 的所有信贷账户
if sample_apply_id:
    print(f"示例2：查看 apply_id={sample_apply_id} 的所有信贷账户")
    print("="*60)
    sample_creditos = parsed_data["creditos"][parsed_data["creditos"]["apply_id"] == sample_apply_id]
    print(f"共 {len(sample_creditos)} 个信贷账户")
    sample_creditos

### 8.3 统计各机构的查询次数

In [ ]:
# 示例3：统计各机构的查询次数
if len(parsed_data["consultas"]) > 0 and "nombreOtorgante" in parsed_data["consultas"].columns:
    print("示例3：统计各机构的查询次数（Top 20）")
    print("="*60)
    otorgante_stats = parsed_data["consultas"]["nombreOtorgante"].value_counts().head(20)
    otorgante_stats

### 8.4 统计各账户类型的数量

In [ ]:
# 示例4：统计各账户类型的数量
if len(parsed_data["creditos"]) > 0 and "tipoCuenta" in parsed_data["creditos"].columns:
    print("示例4：统计各账户类型的数量")
    print("="*60)
    tipo_cuenta_stats = parsed_data["creditos"]["tipoCuenta"].value_counts()
    tipo_cuenta_stats

### 8.5 查看工作记录的薪资分布

In [ ]:
# 示例5：查看工作记录的薪资分布
if len(parsed_data["empleos"]) > 0 and "salarioMensual" in parsed_data["empleos"].columns:
    print("示例5：工作记录的薪资分布")
    print("="*60)
    salary_stats = parsed_data["empleos"]["salarioMensual"].describe()
    salary_stats

## 9. 输出到文件（可选）

### 9.1 输出到 CSV

In [ ]:
# 输出到 CSV
if WRITE_CSV:
    print("\n[INFO] 正在输出到 CSV 文件...")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    files_written = []
    
    for name, df in parsed_data.items():
        if len(df) > 0:
            output_path = OUTPUT_DIR / f"{name}_{timestamp}.csv"
            print(f"  正在写入: {name} ({len(df):,} 行)...")
            df.to_csv(output_path, index=False, encoding="utf-8-sig")
            files_written.append(output_path)
            print(f"  ✓ {output_path}")
    
    print(f"\n[INFO] CSV 文件输出完成，共 {len(files_written)} 个文件")
    print(f"[INFO] 输出目录: {OUTPUT_DIR.absolute()}")
else:
    print("\n[INFO] WRITE_CSV=False，跳过 CSV 输出")

### 9.2 输出到 Excel

In [ ]:
# 输出到 Excel
if WRITE_EXCEL:
    print("\n[INFO] 正在输出到 Excel 文件...")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    files_written = []
    
    for name, df in parsed_data.items():
        if len(df) > 0:
            output_path = OUTPUT_DIR / f"{name}_{timestamp}.xlsx"
            print(f"  正在写入: {name} ({len(df)} 行)...")
            df.to_excel(output_path, index=False, engine="openpyxl")
            files_written.append(output_path)
            print(f"  ✓ {output_path}")
    
    print(f"\n[INFO] Excel 文件输出完成，共 {len(files_written)} 个文件")
    print(f"[INFO] 输出目录: {OUTPUT_DIR.absolute()}")
else:
    print("\n[INFO] WRITE_EXCEL=False，跳过 Excel 输出")

## 完成

解析完成！你可以：

1. 在上面的单元格中查看各板块的数据
2. 如果开启了 `WRITE_EXCEL=True`，可以在 `outputs/response_body_flat/` 目录下找到输出的 Excel 文件
3. 使用上面的查询示例来探索数据
4. 根据需要修改查询条件，进行自定义分析

### 提示

- 如果要处理全部数据，设置 `MAX_SAMPLES = None`
- 如果要快速测试，设置 `MAX_SAMPLES = 10`
- 如果要输出文件，设置 `WRITE_EXCEL = True`